# Alpamayo 1.5: Final Front Camera Projection Notebook

Daily-use final version.

Features:
1. Uses `Front_camera_original_intrinsic.txt` by default
2. Overlays Chain-of-Causation (CoT) on the projected image
3. Shows a clean two-panel view:
   - left: front camera image with GT / Pred projection
   - right: BEV trajectory plot
4. Supports both:
   - single-window testing
   - sliding-window multi-sample testing
5. Keeps output concise for daily usage
6. Prints BEV GT / Pred coordinates for the selected window at the end, without scientific notation

Geometry assumptions used here:
- plotting frame -> undo eval rotation -> raw t0-local
- raw t0-local -> candidate ego/body frame
- ego/body frame -> camera frame using inverse-selected extrinsics
- camera frame -> image plane using `Front_camera_original_intrinsic.txt`

Candidate body mapping:
- local x = lateral
- local y = forward
- local z = up
- body x = forward = local y
- body y = left    = -local x
- body z = up      = -local z

Projection convention:
- camera X = horizontal
- camera Y is flipped before image projection
- camera Z = forward depth


In [ ]:
import os
import sys
import textwrap
from pathlib import Path

repo_root = Path.cwd().resolve().parent
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import av
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import BitsAndBytesConfig

from alpamayo1_5 import helper
from alpamayo1_5.load_offline_dataset import (
    load_offline_dataset,
    get_or_build_offline_clip_cache,
)
from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5
from alpamayo1_5.offline_eval_utils import (
    set_reproducible_seeds,
    rotate_xyz_xy_plane,
    run_offline_inference_window,
    _compute_adaptive_xy_limits,
)

print('repo_root =', repo_root)
print('src_path =', src_path)


In [ ]:
# ===== Reproducibility =====
SEED = 42
set_reproducible_seeds(SEED)

# ===== Paths =====
CLIP_DIR = Path('/workspace/dataset/')
CALIB_DIR = repo_root / 'calibration'
MODEL_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Alpamayo-1.5-10B/snapshots/f11cd25b758ab560114019b555dde2a8b92d88b4')
PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b')
COSMOS_PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Cosmos-Reason2-8B/snapshots/f715d875a8a99a0a2b65aa28633afd9417e46bd9')

# ===== Calibration =====
FRONT_K_ORIG_PATH = CALIB_DIR / 'Front_camera_original_intrinsic.txt'
FRONT_EXT_PATH = CALIB_DIR / 'Front_camera_extrinsics.txt'
FRONT_DIST_PATH = CALIB_DIR / 'Front_camera_distortion.txt'

# ===== Camera =====
TARGET_CAMERA_FILENAME = 'Front_camera.mp4'

# ===== Inference config =====
DEVICE = 'cuda'
NUM_HISTORY_STEPS = 16
NUM_FUTURE_STEPS = 64
TIME_STEP = 0.1
NUM_FRAMES = 4
FPS = 30.0
FRAME0_GPS_TIME_SOD = 175484.98
NUM_TRAJ_SAMPLES = 1
TEMPERATURE = 0.6
TOP_P = 0.98
MAX_GENERATION_LENGTH = 256
IMAGE_SIZE = (448, 800)
EVAL_XY_ROTATION_DEG = -90.0
FRONT_FRAME_T = -1

# ===== CoT overlay formatting =====
COT_WRAP_WIDTH = 72
COT_MAX_CHARS = 1000

# ===== Run mode =====
RUN_MODE = 'sliding'  # 'single' or 'sliding'
SINGLE_T0_SOD = 175500.23

# ===== Sliding-window settings =====
# If SLIDING_T0_SOD_LIST is non-empty, it takes priority.
SLIDING_T0_SOD_LIST = []
SLIDING_START_T0_SOD = 175500.23
SLIDING_END_T0_SOD = 175503.23
SLIDING_STEP_SEC = 0.5
MAX_SLIDING_WINDOWS = None  # e.g. 10 for quick debugging

# ===== Viewer =====
WINDOW_INDEX_TO_VIEW = 0

os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'] = str(COSMOS_PROCESSOR_PATH)

print('SEED =', SEED)
print('RUN_MODE =', RUN_MODE)
print('CLIP_DIR =', CLIP_DIR)
print('CALIB_DIR =', CALIB_DIR)
print('TARGET_CAMERA_FILENAME =', TARGET_CAMERA_FILENAME)


### Helper functions


In [ ]:
def load_matrix_txt(path: Path):
    arr = np.loadtxt(path, dtype=np.float64)
    print(f'Loaded {path.name}: shape={arr.shape}')
    return arr


def transform_points_homogeneous(points_xyz: np.ndarray, T: np.ndarray) -> np.ndarray:
    pts_h = np.concatenate(
        [points_xyz.astype(np.float64), np.ones((len(points_xyz), 1), dtype=np.float64)],
        axis=1,
    )
    out_h = (T @ pts_h.T).T
    return out_h[:, :3]


def project_points_pinhole(points_cam: np.ndarray, K: np.ndarray):
    X = points_cam[:, 0]
    Y = -points_cam[:, 1]
    Z = points_cam[:, 2]

    valid = Z > 1e-6
    u = np.full_like(X, np.nan, dtype=np.float64)
    v = np.full_like(Y, np.nan, dtype=np.float64)

    u[valid] = K[0, 0] * X[valid] / Z[valid] + K[0, 2]
    v[valid] = K[1, 1] * Y[valid] / Z[valid] + K[1, 2]
    return np.stack([u, v], axis=-1), valid


def decode_single_frame(video_path: Path, frame_index: int) -> np.ndarray:
    with av.open(str(video_path)) as container:
        stream = container.streams.video[0]
        for idx, frame in enumerate(container.decode(stream)):
            if idx == frame_index:
                return frame.to_ndarray(format='rgb24')
    raise IndexError(f'Frame index {frame_index} out of range for {video_path}')


def pick_camera_row_by_exact_filename(clip_dir: Path, filename: str):
    camera_paths = helper.discover_offline_camera_files(clip_dir)
    camera_paths_sorted = sorted(camera_paths, key=lambda p: helper.infer_camera_index(p.name))
    matches = [(row, p) for row, p in enumerate(camera_paths_sorted) if p.name == filename]

    if len(matches) == 0:
        raise ValueError(f'No camera file matched exact filename={filename!r}')
    if len(matches) > 1:
        raise ValueError(f'Multiple camera files matched exact filename={filename!r}')

    row, path = matches[0]
    return row, path


def choose_reasonable_extrinsic_direction(points_body: np.ndarray, T_candidate: np.ndarray, K: np.ndarray, image_shape_hw):
    h, w = image_shape_hw

    def score(T):
        pts_cam = transform_points_homogeneous(points_body, T)
        uv, valid_z = project_points_pinhole(pts_cam, K)
        valid_uv = (
            valid_z
            & np.isfinite(uv[:, 0])
            & np.isfinite(uv[:, 1])
            & (uv[:, 0] >= 0)
            & (uv[:, 0] < w)
            & (uv[:, 1] >= 0)
            & (uv[:, 1] < h)
        )
        return int(valid_uv.sum()), uv, pts_cam, valid_uv

    score_fwd, uv_fwd, pts_cam_fwd, valid_fwd = score(T_candidate)
    score_inv, uv_inv, pts_cam_inv, valid_inv = score(np.linalg.inv(T_candidate))

    if score_inv > score_fwd:
        return np.linalg.inv(T_candidate), uv_inv, pts_cam_inv, valid_inv, 'inverse'
    return T_candidate, uv_fwd, pts_cam_fwd, valid_fwd, 'forward'


def plotting_frame_to_t0_local(xyz_plot: np.ndarray, eval_xy_rotation_deg: float) -> np.ndarray:
    return rotate_xyz_xy_plane(xyz_plot, -eval_xy_rotation_deg)


def convert_t0_local_to_camera_body_frame(xyz_t0_local: np.ndarray) -> np.ndarray:
    R = np.array([
        [0.0,  1.0,  0.0],
        [-1.0, 0.0,  0.0],
        [0.0,  0.0, -1.0],
    ], dtype=np.float64)
    return xyz_t0_local @ R.T


def format_cot_text(cot_value, wrap_width=72, max_chars=1000):
    if cot_value is None:
        return 'Chain-of-Causation: <empty>'
    text = str(cot_value).strip()
    if len(text) > max_chars:
        text = text[:max_chars] + ' ...'
    return 'Chain-of-Causation:\n' + textwrap.fill(text, width=wrap_width)


def draw_projection_panel(ax, image, uv_gt, valid_gt, uv_pred, valid_pred, title, cot_text, image_w, image_h):
    ax.imshow(image)

    if valid_gt.any():
        ax.plot(
            uv_gt[valid_gt, 0],
            uv_gt[valid_gt, 1],
            'k-o',
            linewidth=2.5,
            markersize=4,
            label='GT',
        )

    if valid_pred.any():
        ax.plot(
            uv_pred[valid_pred, 0],
            uv_pred[valid_pred, 1],
            'r-o',
            linewidth=2.5,
            markersize=4,
            label='Pred',
        )

    ax.set_xlim(0, image_w)
    ax.set_ylim(image_h, 0)
    ax.set_title(title)
    ax.legend(loc='upper right')

    ax.text(
        0.02,
        0.98,
        cot_text,
        transform=ax.transAxes,
        va='top',
        ha='left',
        fontsize=9,
        color='white',
        bbox=dict(boxstyle='round', facecolor='black', alpha=0.65),
    )


def build_t0_list(run_mode, single_t0_sod, sliding_t0_sod_list, sliding_start, sliding_end, sliding_step, max_windows=None):
    if run_mode == 'single':
        return [float(single_t0_sod)]

    if sliding_t0_sod_list:
        t0_list = [float(x) for x in sliding_t0_sod_list]
    else:
        t0_list = list(np.arange(sliding_start, sliding_end + 1e-9, sliding_step, dtype=np.float64))

    if max_windows is not None:
        t0_list = t0_list[:max_windows]
    return t0_list


def run_projection_window(t0_sod, *, model, processor, K_front_orig, T_front, clip_dir, front_cam_row, front_video_path):
    data = load_offline_dataset(
        clip_dir=clip_dir,
        t0_sod=float(t0_sod),
        num_history_steps=NUM_HISTORY_STEPS,
        num_future_steps=NUM_FUTURE_STEPS,
        time_step=TIME_STEP,
        num_frames=NUM_FRAMES,
        fps=FPS,
        frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
        debug=False,
        image_size=IMAGE_SIZE,
    )

    result = run_offline_inference_window(
        data=data,
        model=model,
        processor=processor,
        device=DEVICE,
        top_p=TOP_P,
        temperature=TEMPERATURE,
        num_traj_samples=NUM_TRAJ_SAMPLES,
        max_generation_length=MAX_GENERATION_LENGTH,
        time_step=TIME_STEP,
        eval_xy_rotation_deg=EVAL_XY_ROTATION_DEG,
    )

    gt_xyz_t0_local = plotting_frame_to_t0_local(result['gt_xyz_plot'], EVAL_XY_ROTATION_DEG)
    pred_xyz_t0_local = plotting_frame_to_t0_local(result['pred_best_xyz'], EVAL_XY_ROTATION_DEG)

    gt_xyz_body = convert_t0_local_to_camera_body_frame(gt_xyz_t0_local)
    pred_xyz_body = convert_t0_local_to_camera_body_frame(pred_xyz_t0_local)

    requested_front_frame_idx = int(data['actual_video_frame_indices'][front_cam_row, FRONT_FRAME_T].item())
    requested_front_ts = float(data['absolute_timestamps_sod'][front_cam_row, FRONT_FRAME_T].item())

    front_img_orig = decode_single_frame(front_video_path, requested_front_frame_idx)
    image_h, image_w = front_img_orig.shape[:2]

    T_front_used, _, _, _, extrinsic_mode = choose_reasonable_extrinsic_direction(
        pred_xyz_body,
        T_front,
        K_front_orig,
        (image_h, image_w),
    )

    gt_xyz_cam = transform_points_homogeneous(gt_xyz_body, T_front_used)
    pred_xyz_cam = transform_points_homogeneous(pred_xyz_body, T_front_used)

    uv_gt, valid_gt = project_points_pinhole(gt_xyz_cam, K_front_orig)
    uv_pred, valid_pred = project_points_pinhole(pred_xyz_cam, K_front_orig)

    metrics_row = result['df_metrics'].iloc[0].to_dict() if len(result['df_metrics']) else {}

    return {
        't0_sod': float(t0_sod),
        'data': data,
        'result': result,
        'front_img_orig': front_img_orig,
        'image_h': image_h,
        'image_w': image_w,
        'requested_front_frame_idx': requested_front_frame_idx,
        'requested_front_ts': requested_front_ts,
        'extrinsic_mode': extrinsic_mode,
        'uv_gt': uv_gt,
        'valid_gt': valid_gt,
        'uv_pred': uv_pred,
        'valid_pred': valid_pred,
        'metrics_row': metrics_row,
        'cot_text': format_cot_text(result.get('cot_value', ''), wrap_width=COT_WRAP_WIDTH, max_chars=COT_MAX_CHARS),
        'gt_bev': result['gt_xyz_plot'],
        'pred_bev': result['pred_best_xyz'],
        'hist_bev': result['hist_xyz_plot'],
    }


### Load calibration, cache, model, and processor


In [ ]:
for p in [FRONT_K_ORIG_PATH, FRONT_EXT_PATH, FRONT_DIST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f'Missing calibration file: {p}')

K_front_orig = load_matrix_txt(FRONT_K_ORIG_PATH)
T_front = load_matrix_txt(FRONT_EXT_PATH)
dist_front = load_matrix_txt(FRONT_DIST_PATH)

clip_cache = get_or_build_offline_clip_cache(CLIP_DIR, debug=False, force_rebuild=False)
front_cam_row, front_video_path = pick_camera_row_by_exact_filename(CLIP_DIR, TARGET_CAMERA_FILENAME)

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
)

model = Alpamayo1_5.from_pretrained(
    str(MODEL_PATH),
    dtype=torch.bfloat16,
    device_map='cuda:0',
    quantization_config=quantization_config,
)

processor = helper.get_processor(
    model.tokenizer,
    processor_name_or_path=PROCESSOR_PATH,
)

print('Loaded calibration, cache, model, and processor.')
print('front_cam_row =', front_cam_row)
print('front_video_path =', front_video_path)


### Run single window or sliding-window projection


In [ ]:
t0_list = build_t0_list(
    run_mode=RUN_MODE,
    single_t0_sod=SINGLE_T0_SOD,
    sliding_t0_sod_list=SLIDING_T0_SOD_LIST,
    sliding_start=SLIDING_START_T0_SOD,
    sliding_end=SLIDING_END_T0_SOD,
    sliding_step=SLIDING_STEP_SEC,
    max_windows=MAX_SLIDING_WINDOWS,
)

print('num windows =', len(t0_list))
print('first few t0_sod =', t0_list[:5])

window_results = []
for i, t0_sod in enumerate(t0_list, start=1):
    print(f'Running window {i}/{len(t0_list)} | t0_sod={t0_sod:.3f}')
    window_result = run_projection_window(
        t0_sod=t0_sod,
        model=model,
        processor=processor,
        K_front_orig=K_front_orig,
        T_front=T_front,
        clip_dir=CLIP_DIR,
        front_cam_row=front_cam_row,
        front_video_path=front_video_path,
    )
    window_results.append(window_result)

print('Finished all windows.')


### Concise result summary table


In [ ]:
summary_rows = []
for wr in window_results:
    row = {
        't0_sod': wr['t0_sod'],
        'frame_idx': wr['requested_front_frame_idx'],
        'frame_ts': wr['requested_front_ts'],
        'extrinsic_mode': wr['extrinsic_mode'],
        'valid_gt': int(wr['valid_gt'].sum()),
        'valid_pred': int(wr['valid_pred'].sum()),
    }
    row.update(wr['metrics_row'])
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
display(df_summary)


### Visualize one selected window


In [ ]:
wr = window_results[WINDOW_INDEX_TO_VIEW]

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

draw_projection_panel(
    ax=axes[0],
    image=wr['front_img_orig'],
    uv_gt=wr['uv_gt'],
    valid_gt=wr['valid_gt'],
    uv_pred=wr['uv_pred'],
    valid_pred=wr['valid_pred'],
    title=f"Front projection | t0={wr['t0_sod']:.3f} | frame={wr['requested_front_frame_idx']} | extrinsic={wr['extrinsic_mode']}",
    cot_text=wr['cot_text'],
    image_w=wr['image_w'],
    image_h=wr['image_h'],
)

ax = axes[1]
xlim, ylim = _compute_adaptive_xy_limits(wr['hist_bev'], wr['gt_bev'], wr['pred_bev'], min_span=20.0, pad_ratio=0.12, pad_min=2.0)
ax.plot(wr['hist_bev'][:, 0], wr['hist_bev'][:, 1], 'b-o', linewidth=2, markersize=3, label='History')
ax.plot(wr['gt_bev'][:, 0], wr['gt_bev'][:, 1], 'k-o', linewidth=2.5, markersize=3.5, label='GT')
ax.plot(wr['pred_bev'][:, 0], wr['pred_bev'][:, 1], 'r-o', linewidth=2.5, markersize=3.5, label='Pred')
ax.scatter([0.0], [0.0], c='red', marker='x', s=120, label='t0 ego')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('BEV trajectory plot')
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal', adjustable='box')
ax.legend()

plt.tight_layout()
plt.show()


### Optional debug tables for the selected window


In [ ]:
wr = window_results[WINDOW_INDEX_TO_VIEW]

df_gt_proj = pd.DataFrame({
    'u': wr['uv_gt'][:, 0],
    'v': wr['uv_gt'][:, 1],
    'valid': wr['valid_gt'],
})

df_pred_proj = pd.DataFrame({
    'u': wr['uv_pred'][:, 0],
    'v': wr['uv_pred'][:, 1],
    'valid': wr['valid_pred'],
})

print('[GT projection points]')
display(df_gt_proj)

print('[Pred projection points]')
display(df_pred_proj)


### Print BEV GT / Pred coordinates for the selected window (no scientific notation)


In [ ]:
wr = window_results[WINDOW_INDEX_TO_VIEW]

np.set_printoptions(suppress=True, precision=6)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')

df_gt_bev = pd.DataFrame(wr['gt_bev'], columns=['x', 'y', 'z'])
df_pred_bev = pd.DataFrame(wr['pred_bev'], columns=['x', 'y', 'z'])

print(f"[Selected window] index={WINDOW_INDEX_TO_VIEW}, t0_sod={wr['t0_sod']:.6f}")
print('\n[BEV GT trajectory coordinates]')
display(df_gt_bev)

print('\n[BEV Pred trajectory coordinates]')
display(df_pred_bev)
